In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/mlss-sp-25/sample_submission.csv
/kaggle/input/mlss-sp-25/train.csv
/kaggle/input/mlss-sp-25/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/mlss-sp-25/train.csv")
test = pd.read_csv("/kaggle/input/mlss-sp-25/test.csv")
sample = pd.read_csv("/kaggle/input/mlss-sp-25/sample_submission.csv")
ID = test["object_id"]
train = train.drop(["object_id", 'date'], axis = 1)
test = test.drop(['date', "object_id"], axis = 1)

In [3]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [4]:
train

,transaction_currency,height_cm,width_cm,location,category,n_previous_owners,creation_year,is_titled,signed,dated,n_exhibited,artist_birth_year,artist_death_year,estimate,price_realized
0,USD,67.3,55.9,New York,Paintings,4.0,1996.0,NaN,True,True,0,1937.0,NaN,15000.00,9375.000
1,USD,122.2,122.2,New York,Paintings,1.0,NaN,False,True,True,0,1952.0,NaN,2500.00,3750.000
2,USD,15.2,10.2,New York,Paintings,2.0,NaN,True,True,True,2,1958.0,NaN,6000.00,11250.000
3,USD,58.4,67.3,New York,Paintings,2.0,NaN,False,True,True,0,1973.0,NaN,5000.00,9375.000
4,USD,68.6,52.7,New York,Paintings,1.0,1957.0,NaN,True,False,0,1917.0,NaN,25000.00,15000.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105896,EUR,83.3,65.4,Paris,Paintings,1.0,NaN,NaN,False,False,5,1864.0,1956.0,19724.25,30995.250
105897,EUR,14.9,26.8,Paris,Paintings,NaN,NaN,NaN,False,False,0,1816.0,1878.0,5635.50,5353.725
105898,EUR,22.1,17,Paris,Paintings,NaN,NaN,NaN,False,False,0,1851.0,1929.0,1127.10,1127.100
105899,EUR,49.9,70.9,Paris,Paintings,NaN,NaN,NaN,False,False,0,1624.0,1685.0,14088.75,14088.750


In [5]:
train.isnull().sum()

transaction_currency        0
height_cm               21700
width_cm                21700
location                 1879
category                    0
n_previous_owners       42334
creation_year           75552
is_titled               88123
signed                      0
dated                       0
n_exhibited                 0
artist_birth_year       16253
artist_death_year       32964
estimate                   84
price_realized              0
dtype: int64

In [6]:
train.shape

(105901, 15)

In [7]:
test.isnull().sum()

transaction_currency        0
height_cm                4173
width_cm                 4173
location                    0
category                    0
n_previous_owners        9077
creation_year           27382
is_titled               40819
signed                      0
dated                       0
n_exhibited                 0
artist_birth_year        4967
artist_death_year       16044
estimate                   82
dtype: int64

In [8]:
train = train.drop(['creation_year','is_titled'], axis = 1)
test = test.drop(['creation_year','is_titled'], axis = 1)

In [9]:
mode_imputatipn_col = ['location', 'estimate']
from sklearn.impute import SimpleImputer
mode_imputer = SimpleImputer(strategy = 'most_frequent')
train[mode_imputatipn_col] = mode_imputer.fit_transform(train[mode_imputatipn_col])
test[mode_imputatipn_col] = mode_imputer.transform(test[mode_imputatipn_col])

In [10]:
def clean_numeric(value):
    try:
        return float(value)
    except:
        return None

train = train.copy()
test = test.copy()

train.loc[:, 'height_cm'] = train['height_cm'].apply(clean_numeric)
train.loc[:, 'width_cm'] = train['width_cm'].apply(clean_numeric)
test.loc[:, 'height_cm'] = test['height_cm'].apply(clean_numeric)
test.loc[:, 'width_cm'] = test['width_cm'].apply(clean_numeric)

In [11]:
mean_imputatipn_col = ['artist_birth_year', 'height_cm', 'width_cm', 'n_previous_owners', 'artist_death_year']
from sklearn.impute import SimpleImputer
mean_imputer = SimpleImputer(strategy = 'mean')
train[mean_imputatipn_col] = mean_imputer.fit_transform(train[mean_imputatipn_col])
test[mean_imputatipn_col] = mean_imputer.transform(test[mean_imputatipn_col])

In [12]:
train.isnull().sum().sum()

0

In [13]:
test.isnull().sum().sum()

0

In [14]:
catagorical_cols = ['category', 'signed', 'dated', 'location', 'transaction_currency']
train_dummies = pd.get_dummies(train[catagorical_cols], drop_first = True).astype(int)
test_dummies = pd.get_dummies(test[catagorical_cols], drop_first = True).astype(int)

In [15]:
test_dummies = test_dummies.reindex(columns = train_dummies.columns, fill_value = 0)

In [16]:
train

,transaction_currency,height_cm,width_cm,location,category,n_previous_owners,signed,dated,n_exhibited,artist_birth_year,artist_death_year,estimate,price_realized
0,USD,67.3,55.9,New York,Paintings,4.00000,True,True,0,1937.000000,1913.450965,15000.0,9375.000
1,USD,122.2,122.2,New York,Paintings,1.00000,True,True,0,1952.000000,1913.450965,2500.0,3750.000
2,USD,15.2,10.2,New York,Paintings,2.00000,True,True,2,1958.000000,1913.450965,6000.0,11250.000
3,USD,58.4,67.3,New York,Paintings,2.00000,True,True,0,1973.000000,1913.450965,5000.0,9375.000
4,USD,68.6,52.7,New York,Paintings,1.00000,True,False,0,1917.000000,1913.450965,25000.0,15000.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
105896,EUR,83.3,65.4,Paris,Paintings,1.00000,False,False,5,1864.000000,1956.000000,19724.25,30995.250
105897,EUR,14.9,26.8,Paris,Paintings,2.25831,False,False,0,1816.000000,1878.000000,5635.5,5353.725
105898,EUR,22.1,17.0,Paris,Paintings,2.25831,False,False,0,1851.000000,1929.000000,1127.1,1127.100
105899,EUR,49.9,70.9,Paris,Paintings,2.25831,False,False,0,1624.000000,1685.000000,14088.75,14088.750


In [17]:
train = pd.concat([train.drop(catagorical_cols, axis = 1), train_dummies], axis = 1)
test = pd.concat([test.drop(catagorical_cols, axis = 1), test_dummies], axis = 1)

In [18]:
X_train = train.drop('price_realized', axis = 1)
y_train = train['price_realized']

In [19]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train_scaled = scaler.fit_transform(X_train)
test_scaled = scaler.transform(test)

In [20]:
X_train = train.drop('price_realized', axis = 1)
y_train = train['price_realized']

In [21]:
X_train.shape

(105901, 25)

In [22]:
# Apply PCA
from sklearn.decomposition import PCA
pca = PCA()
X_train_pca = pca.fit_transform(X_train)

In [23]:
X_train_pca

array([[-8.42036749e+03, -1.21924356e+05, -1.58081524e+01, ...,
         2.48263150e-16,  9.97853681e-18, -8.22192019e-20],
       [-8.36236367e+03, -1.34424341e+05,  5.05591990e+01, ...,
        -7.84926639e-17, -1.34038886e-17, -2.88483135e-19],
       [-8.47023274e+03, -1.30924369e+05, -6.14536808e+01, ...,
         3.32512080e-14,  2.18722185e-16, -7.89828856e-19],
       ...,
       [-8.46212279e+03, -1.35797267e+05, -5.46933416e+01, ...,
        -2.24835445e-17,  2.75027878e-19,  2.71358713e-21],
       [-8.43754126e+03, -1.22835611e+05, -9.87033451e-01, ...,
        -1.28063413e-17, -4.23223951e-19,  1.70912355e-20],
       [-8.47243953e+03, -1.08746869e+05, -5.58145503e+01, ...,
         3.70707743e-18,  3.48890549e-19,  5.38636255e-20]])

In [24]:
X_train_pca.shape

(105901, 25)

In [25]:
# Compute explained variance
explained_variance = np.cumsum(pca.explained_variance_ratio_)
explained_variance

array([0.83916152, 0.99999982, 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ])

In [26]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_percentage_error

In [27]:
# Cross-validation to find the best M using MAPE
kf = KFold(n_splits=5, shuffle=True, random_state=42)

best_score = float("inf")  # MAPE should be minimized
best_m = 1

for m in range(1, X_train.shape[1] + 1):
    X_train_reduced = X_train_pca[:, :m]
    
    # Train Ridge Regression on M components
    model = LinearRegression()
    scores = cross_val_score(model, X_train_reduced, y_train, cv=kf, scoring="neg_mean_absolute_percentage_error")
    mean_score = -scores.mean()  # Convert negative value to positive (MAPE should be minimized)
    if mean_score < best_score:
        best_score = mean_score
        best_m = m

In [28]:
# Report variance explained by best M components
print(f"\nThe {best_m} principal components explain {explained_variance[best_m-1]*100:.2f}% of variance in the training set.")


The 8 principal components explain 100.00% of variance in the training set.


In [29]:
# Train final model on the best M components
X_train_final = X_train_pca[:, :best_m]
model_final = LinearRegression()
model_final.fit(X_train_final, y_train)

LinearRegression()

In [30]:
# Find top 2 variables contributing to PC1
pc1_loadings = np.abs(pca.components_[0])
top_2_features = X_train.columns[np.argsort(pc1_loadings)[-2:]].tolist()
print(f"\nTop 2 features contributing to PC1: {top_2_features}")


Top 2 features contributing to PC1: ['estimate', 'height_cm']


In [31]:
# Transform test set & make predictions
X_test_pca = pca.transform(test)[:, :best_m]
y_test_pred = model_final.predict(X_test_pca)

In [32]:
sample

,object_id,price_realized
0,gnite,67887.1425
1,agxtc,3655.0700
2,otrwd,43267.3000
3,yqyhe,39582.9200
4,cpqne,2750.6000
...,...,...
52946,yhuiq,35266.0000
52947,kwujo,166352.0000
52948,wyapo,43737.1000
52949,ebiyn,8948.5000


In [33]:
submission = pd.DataFrame({"object_id": ID, "price_realized": y_test_pred})

In [34]:
submission

,object_id,price_realized
0,gnite,21461.061625
1,agxtc,99451.776277
2,otrwd,-2780.718851
3,yqyhe,29617.317581
4,cpqne,62188.349346
...,...,...
52946,yhuiq,42555.447871
52947,kwujo,343367.685927
52948,wyapo,39555.910258
52949,ebiyn,8579.275223


In [35]:
submission.to_csv("submission.csv", index=False)